In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
import re
import sys
from PIL import Image
from torchvision import transforms

# ============================================================================
# 1. SETUP & IMPORTS (Matching your project structure)
# ============================================================================
project_root = Path(__file__).resolve().parent.parent.parent
sys.path.insert(0, str(project_root))

from main.Models import (
    VGG16AuthenticityPredictor,
    VGG19AuthenticityPredictor,
    ResNet152AuthenticityPredictor,
    DenseNet161AuthenticityPredictor,
    EfficientNetB3AuthenticityPredictor,
    BarlowTwinsAuthenticityPredictor,
)

# Required for stacking ensemble mode
class StackingMetaLearner(nn.Module):
    def __init__(self, num_base_models: int):
        super().__init__()
        self.fc = nn.Linear(num_base_models, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

# ============================================================================
# 2. PREDICTION SCRIPT
# ============================================================================

def get_image_transform(is_densenet: bool):
    """
    Returns the appropriate transform. 
    Based on experiment 3d, DenseNet uses 300x300, others use 224x224.
    """
    img_size = 300 if is_densenet else 224
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])

def predict_single_image(image_path: str, weights_dir: str, ensemble_mode="stacking", device="cuda"):
    """
    Given an image path and the directory containing pruned weights, 
    returns a single scalar prediction.
    """
    weights_path = Path(weights_dir)
    img_path = Path(image_path)
    
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found at {img_path}")

    MODEL_REGISTRY = {
        "vgg16": {"class": VGG16AuthenticityPredictor, "is_densenet": False},
        "vgg19": {"class": VGG19AuthenticityPredictor, "is_densenet": False},
        "resnet152": {"class": ResNet152AuthenticityPredictor, "is_densenet": False},
        "densenet161": {"class": DenseNet161AuthenticityPredictor, "is_densenet": True},
        "efficientnetb3": {"class": EfficientNetB3AuthenticityPredictor, "is_densenet": False},
        "barlowtwins": {"class": BarlowTwinsAuthenticityPredictor, "is_densenet": False},
    }

    # 1. Find all pruned weights exactly as in experiment_3c
    all_pruned_files = sorted(weights_path.glob("*_exp3b_*_greedy_pruned.pth"))
    
    ordered_pairs = []
    for p in all_pruned_files:
        match = re.match(r"^([a-z0-9]+)_exp3b_variant(\d+)_greedy_pruned\.pth$", p.name)
        if match and match.group(1) in MODEL_REGISTRY:
            ordered_pairs.append((match.group(1), p))

    if not ordered_pairs:
        raise FileNotFoundError(f"No pruned weights found in {weights_path}")

    # 2. Load and prep the raw image
    raw_image = Image.open(img_path).convert("RGB")
    all_preds = []

    # 3. Get predictions from all base models
    print(f"Generating predictions from {len(ordered_pairs)} base models...")
    with torch.no_grad():
        for model_name, weight_file in ordered_pairs:
            config = MODEL_REGISTRY[model_name]
            
            # Apply specific transform and add batch dimension: [1, C, H, W]
            transform = get_image_transform(config["is_densenet"])
            img_tensor = transform(raw_image).unsqueeze(0).to(device)

            # Initialize and load model weights
            model = config["class"](freeze_backbone=False)
            model.load_state_dict(torch.load(weight_file, map_location=device, weights_only=True))
            model.to(device)
            model.eval()

            # Forward pass (model returns outputs, features)
            outputs, _ = model(img_tensor)
            all_preds.append(outputs.cpu().squeeze())

            # Free memory
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Shape: (num_models,)
    preds_tensor = torch.stack(all_preds)

    # 4. Aggregate via Ensemble Mode
    if ensemble_mode == "bagging":
        final_prediction = torch.mean(preds_tensor).item()
        
    elif ensemble_mode == "stacking":
        meta_weights_path = weights_path / "stacking_meta_weights.pth"
        if not meta_weights_path.exists():
            raise FileNotFoundError("Stacking weights missing. Run experiment 3c with stacking first.")
            
        meta = StackingMetaLearner(num_base_models=len(ordered_pairs))
        meta.load_state_dict(torch.load(meta_weights_path, map_location="cpu", weights_only=True))
        meta.eval()

        with torch.no_grad():
            # Meta learner expects batch dimension: [1, num_models]
            final_prediction = meta(preds_tensor.unsqueeze(0)).item()
            
    else:
        raise ValueError(f"Unknown ensemble mode: {ensemble_mode}")

    return final_prediction

# ============================================================================
# 3. USAGE EXAMPLE
# ============================================================================
if __name__ == "__main__":
    IMAGE_PATH = "./my_test_folder/sample_image.jpg"
    WEIGHTS_DIR = str(Path(__file__).resolve().parent / "tmp_Outputs" / "Experiment_3_ensemble" / "Weights")
    
    # Choose "stacking" or "bagging"
    prediction = predict_single_image(
        image_path=IMAGE_PATH,
        weights_dir=WEIGHTS_DIR,
        ensemble_mode="stacking",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    
    print(f"\nFinal Authenticity Prediction: {prediction:.4f}")